In [23]:
# DON'T MODIFY THIS CELL
using Lattices
using ChainRulesCore
using ExponentialUtilities
using LinearAlgebra
using Combinatorics
using SparseArrays
using Plots
import Graphs
using LaTeXStrings
using Statistics
using Random
using Zygote
using Optimization, OptimizationOptimisers
using JSON
using OptimizationOptimJL
using JLD2

include("ed_objects.jl")
include("utility_functions.jl")




load_saved_dict

In [31]:
## DON'T MODIFY THIS CELL ##

function make_hermitian(A::SparseMatrixCSC)
    # acts similar to Hermitian(A) but is when only one of A[i,j] and A[j,i] are non-zero
    # This function doesn't override non-zero values with zero values like Hermitian(A) can
    I, J, V = findnz(A)
    return sparse(
        vcat(I, J),
        vcat(J, I),
        vcat(V, conj.(V)),
        size(A, 1), size(A, 2)
    )
end
function make_antihermitian(A::SparseMatrixCSC)
    # acts similar to Hermitian(A) but is when only one of A[i,j] and A[j,i] are non-zero
    # This function doesn't override non-zero values with zero values like Hermitian(A) can

    I, J, V = findnz(A)
    return sparse(
        vcat(I, J),
        vcat(J, I),
        vcat(V, -conj.(V)),
        size(A, 1), size(A, 2)
    )
end


function compute_jw_sign(
    conf::Tuple{Set{T},Set{T}},
    sorted_sites::Vector{T},
    ops::Vector{Tuple{T,Int,Symbol}}
) where T
    # computes the sign for the term given by ops (in second quantized), associated with the 
    # configuration conf.

    # Full JW order over sites and spins
    jw_order = [(s, σ) for s in sorted_sites for σ in (1, 2)]

    # Map each mode to its index in JW order
    jw_index = Dict{Tuple{T,Int},Int}((sσ, i) for (i, sσ) in enumerate(jw_order))

    # Initial occupation vector (ordered)
    occupied_modes = Set{Tuple{T,Int}}()
    for (s, σ) in jw_order
        if s ∈ conf[σ]
            push!(occupied_modes, (s, σ))
        end
    end

    # Compute sign by counting how many occupied modes come before each operator
    sign = 1

    # Process from right to left (annihilate first)
    for (site, spin, op) in reverse(ops)
        mode = (site, spin)
        idx = jw_index[mode]

        # Count how many occupied modes come *before* this mode
        n_occupied_before = count(m -> jw_index[m] < idx, occupied_modes)
        sign *= (-1)^n_occupied_before

        # Update occupation based on op
        if op == :annihilate
            # Fermion is removed — no longer present
            delete!(occupied_modes, mode)
        elseif op == :create
            # Fermion is added — affects future operators
            push!(occupied_modes, mode)
        else
            error("Unknown operator: $op")
        end
    end

    return sign
end

"""
function which takes in a set of keys corresponding to different operators, and outputs the 
    information required to construct a sparse matrix, without specifying what the coefficient
    values are. This includes row, column indices, plus the sign at that element, and the operator
    the matrix element is associated with.
"""
function build_nth_order_sparse(
    t_keys::AbstractVector,
    indexer::CombinationIndexer{T},
    ::Type{U}=Float64;
    skip_lower_triangular::Bool=false
) where {T,U<:Number}
    sorted_sites = sort(indexer.a)
    rows = Int[]
    cols = Int[]
    signs = U[]
    ops_list = Vector{Vector{Tuple{T,Int,Symbol}}}()

    for (i1, conf) in enumerate(indexer.inv_comb_dict)
        for ops in t_keys
            # Clone the config
            conf_new = [copy(conf[1]), copy(conf[2])]
            valid = true

            # Apply operators
            for (site, spin, op) in reverse(ops)
                if op == :annihilate
                    if site ∉ conf_new[spin]
                        valid = false
                        break
                    end
                    delete!(conf_new[spin], site)
                elseif op == :create
                    if site ∈ conf_new[spin]
                        valid = false
                        break
                    end
                    push!(conf_new[spin], site)
                else
                    error("Invalid operator symbol: $op")
                end
            end

            if !valid
                continue
            end

            i2 = index(indexer, conf_new[1], conf_new[2])
            if skip_lower_triangular && i1 > i2 #only considering upper diagonal so ensure hermiticity
                continue
            end
            s = compute_jw_sign(conf, sorted_sites, ops)
            push!(rows, i1)
            push!(cols, i2)
            push!(signs, s)
            push!(ops_list, ops)
        end
    end

    return rows, cols, signs, ops_list
end

"""
based on the indexer defining the basis, this function creates a set of keys (not ordered in any particular way) 
    associated list all possible operators that can be put in the exponential.
"""
function build_nth_order_operator(n::Int, indexer::CombinationIndexer; omit_H_conj::Bool=false, 
    conserve_spin::Bool=false, conserve_momentum::Bool=false)
    # function creates a dictionary of free parameters in the form of a dictionary. 
    # when spin is conserved, the Hilbert space is smaller, so a restricted number of coefficients are possible. The rest aren't filled in
    # When hermiticity is forced, we only need to worry about upper diagonal elements. The rest can be filled in afterward

    t_keys = Set{Vector{Tuple{Coordinate{2,Int64},Int,Symbol}}}()
    site_list = sort(indexer.a) #ensuring normal ordering
    all_ops(label) = combinations([(s, σ, label) for s in site_list for σ in 1:2], n)
    equal_spin(create, annihilate) = sum((σ * 2 - 3) for (s, σ, _) in create) == sum((σ * 2 - 3) for (s, σ, _) in annihilate)
    geq_ops(create, annihilate) = [(s.coordinates..., σ) for (s, σ, _) in create] <= [(s.coordinates..., σ) for (s, σ, _) in annihilate]

    for (ops_create, ops_annihilate) in Iterators.product(all_ops(:create), all_ops(:annihilate))
        key = [ops_create; ops_annihilate]

        # We must conserve momentum if the user specified conserve_momentum=true
        # OR if the indexer is explicitly restricted to a momentum sector (which means non-conserving ops will jump out of the Hilbert space)
        must_conserve_momentum = conserve_momentum || (!isnothing(indexer.k) && !isnothing(indexer.lattice_dims))

        if must_conserve_momentum
            tot_k = zeros(Int, length(indexer.lattice_dims))
            for (s, σ, _) in ops_create
                tot_k .+= (s.coordinates .- 1)
            end
            for (s, σ, _) in ops_annihilate
                tot_k .-= (s.coordinates .- 1)
            end
            tot_k = tot_k .% indexer.lattice_dims
            is_momentum_conserved = all(tot_k .== 0)
        else
            is_momentum_conserved = true
        end

        if (!omit_H_conj || geq_ops(ops_create, ops_annihilate)) && (!conserve_spin || equal_spin(ops_create, ops_annihilate)) && is_momentum_conserved
            if key ∉ t_keys
                push!(t_keys, key)
            end
        end
    end
    return collect(t_keys)
end


build_nth_order_operator

In [ ]:
# MUTABLE CELL
""" used to optimize update_values """
function build_param_index_map(
    ops_list::Vector{Vector{Tuple{T,Int,Symbol}}},
    t_keys::Vector{Vector{Tuple{T,Int,Symbol}}}
) where {T}
    # Build reverse lookup: key -> index in t_keys
    key_to_idx = Dict(t_keys[i] => i for i in eachindex(t_keys))
    # For each element in ops_list, find which t_key index it refers to
    return [key_to_idx[ops_list[i]] for i in eachindex(ops_list)]
end

# used to update the values of the sparse matrix efficiently.
function update_values(
    signs::Vector{U},
    param_index_map::Vector{Int},
    t_vals::Vector{V}
) where {U<:Number,V<:Number}
    # it's allowed for length(t_vals) < length(t_keys), but a parameter_mapping to make the difference is required.
    return [t_vals[param_index_map[i]] * signs[i] for i in eachindex(signs)]
end


update_values (generic function with 1 method)

In [27]:
# MUTABLE CELL #
function optimize_unitary(state1::Vector, state2::Vector, indexer::CombinationIndexer;
    maxiters=10, ϵ=1e-5, optimization_scheme::Vector=[1, 2], spin_conserved::Bool=false,
    gradient=:adjoint_gradient, metric_functions::Dict{String,Function}=Dict{String,Function}(),
    antihermitian::Bool=false, optimizer::Union{Symbol,Vector{Symbol}}=:LBFGS, perturb_optimization::Float64=0.001,
    initial_coefficients::Vector{Any}=Any[], initialization_samples::Int=20,
    operator_cache::Dict{Int,Dict{Symbol,Any}}=Dict{Int,Dict{Symbol,Any}}(),
    momentum_basis::Bool=false
)
    # spin_conserved is only true when using (N↑, N↓) and not N
    max_order_scheme = isempty(optimization_scheme) ? 0 : maximum(optimization_scheme)
    max_order_cache = isempty(operator_cache) ? 0 : maximum(keys(operator_cache))
    max_order = max(max_order_scheme, max_order_cache)

    # Initialize return vectors with nothing
    computed_matrices = Vector{Any}(nothing, max_order)
    computed_coefficients = Vector{Any}(nothing, max_order)
    coefficient_labels = Vector{Any}(nothing, max_order)

    # helper for p_args (sum of all OTHER matrices)
    function get_p_args(current_order)
        mats = [m for (i, m) in enumerate(computed_matrices) if i != current_order && !isnothing(m)]
        return isempty(mats) ? nothing : sum(mats)
    end

    dim = length(indexer.inv_comb_dict)
    metrics = Dict{String,Vector{Any}}()
    loss = 1 - abs2(state1' * state2)
    prev_loss = loss
    metrics["loss"] = Float64[loss]
    metrics["other"] = []
    metrics["loss_std"] = Float64[0.0]
    for k in keys(metric_functions)
        metrics[k] = Any[]
    end

    println("Initial loss: $loss")
    println("Dimension: $dim")
    if loss < 1e-15
        println("States are already equal")
        return computed_matrices, coefficient_labels, computed_coefficients, metrics, operator_cache
    end

    # Define a function to ensure operator structure is pre-computed and cached. This operator structure doesn't store any coefficient values.
    function ensure_operator_structure!(order)
        if haskey(operator_cache, order)
            return operator_cache[order]
        end
        # compute operator structure, initial coefficients and operators
        @time t_keys = build_nth_order_operator(order, indexer; omit_H_conj=true, conserve_spin=spin_conserved, conserve_momentum=momentum_basis)
        @time rows, cols, signs, ops_list = build_nth_order_sparse(t_keys, indexer)
        param_index_map = build_param_index_map(ops_list, t_keys)

        # create matrix operators to make gradient computation faster
        ops = []
        for k in collect(t_keys)
            _rows, _cols, _signs, _ = build_nth_order_sparse([k], indexer)
            if antihermitian
                push!(ops, make_antihermitian(sparse(_rows, _cols, _signs, dim, dim)))
            else
                push!(ops, make_hermitian(sparse(_rows, _cols, _signs, dim, dim)))
            end
        end

        cache_entry = Dict(
            :rows => rows,
            :cols => cols,
            :signs => signs,
            :ops_list => ops_list,
            :t_keys => t_keys,
            :param_index_map => param_index_map,
            :ops => ops
        )
        operator_cache[order] = cache_entry
        return cache_entry
    end

    # 2. Main Optimization Scheme Loop
    for order ∈ optimization_scheme
        struct_data = ensure_operator_structure!(order)

        # Extract variables for convenience from struct_data
        rows = struct_data[:rows]
        cols = struct_data[:cols]
        signs = struct_data[:signs]
        ops_list = struct_data[:ops_list]
        t_keys = struct_data[:t_keys]
        param_index_map = struct_data[:param_index_map]
        ops = struct_data[:ops]
        coefficient_labels[order] = t_keys

        function f_adjoint(t_vals, p=nothing)
            return adjoint_loss(t_vals, ops, rows, cols, signs, param_index_map, dim, state2, state1, p, antihermitian)
        end

        tmp_losses = []
        function callback(state, loss_val)
            # state.gradient
            # push!(loss_tracker, loss_val)
            N = 20

            grad_msg = if isnothing(state.grad)
                "gradient=N/A relative-change=N/A curvature=N/A"
            else
                "gradient=$(sum(abs, state.grad)) relative-change=$(sum(state.grad ./ state.u)) curvature=$(sum(state.grad .* state.u))"
            end

            println("loss=$loss_val avg_coef=$(mean(abs.(state.u))) $grad_msg")

            push!(tmp_losses, loss_val)
            if length(tmp_losses) > N && std(tmp_losses[end-N:end]) < 1e-8
                return true
            end
            return false
        end

        if gradient == :gradient
            optf = Optimization.OptimizationFunction(f, Optimization.AutoZygote())
        elseif gradient == :manualgradient
            optf = Optimization.OptimizationFunction(f_nongradient, grad=fg!)
        elseif gradient == :adjoint_gradient
            optf = Optimization.OptimizationFunction(f_adjoint, Optimization.AutoZygote())
        else
            optf = Optimization.OptimizationFunction(f_nongradient)
        end

        optimizers = optimizer isa Vector ? optimizer : [optimizer]

        # runs the optimization for the selection initial coefficients. 
        function execute_optimization(current_t_vals, current_maxiters, _optimizers)
            local_t_vals = copy(current_t_vals)
            local_loss = loss
            local_sol = nothing

            for (optimizer_idx, optimizer_sym) in enumerate(_optimizers)
                # 1. Setup Phase
                # Perturbs the coefficients if the previous optimization reached a bad local minimum, also helps with regularization.
                if optimizer_idx > 1 && perturb_optimization > 1e-9 && mean(abs.(local_t_vals)) > 1e-1
                    local_t_vals = local_t_vals * (1 - perturb_optimization) + perturb_optimization * mean(abs.(local_t_vals)) * (2 * rand(length(local_t_vals)) .- 1)
                end

                p_args = get_p_args(order)

                # Create Problem using global optf (defined outside loop)
                prob = OptimizationProblem(optf, local_t_vals, p_args)

                # Determine Algorithm
                if optimizer_sym == :LBFGS
                    opt_algo = Optim.LBFGS()
                elseif optimizer_sym == :BFGS
                    opt_algo = OptimizationOptimJL.BFGS()
                elseif optimizer_sym == :GradientDescent
                    opt_algo = OptimizationOptimJL.GradientDescent()
                else
                    opt_algo = OptimizationOptimJL.LBFGS()
                end

                # 2. Execution Phase: Single Solve Call
                empty!(tmp_losses)
                println("Solving with $optimizer_sym...")
                @time local_sol = Optimization.solve(prob, opt_algo, maxiters=current_maxiters, callback=callback)

                # 3. Post-Process Phase
                local_t_vals = local_sol.u
                local_loss = local_sol.objective
            end

            return local_sol, local_t_vals, local_loss
        end

        # find values for t_vals. If they don't already exist, they are sampled, and selected based on having a large enough gradient and a low enough loss.
        if length(initial_coefficients) >= order && !isnothing(initial_coefficients[order])
            t_vals = initial_coefficients[order]
        elseif initialization_samples > 0
            println("Sampling $initialization_samples initial configurations over a range of magnitudes...")
            p_args = get_p_args(order)

            good_samples = []
            initial_loss = loss

            log_min = log10(1e-7)
            log_max = log10(1e-1)

            for s in 1:initialization_samples
                mag = 10^(log_min + (log_max - log_min) * rand())

                t_sample = (2 * rand(length(t_keys)) .- 1) * mag


                res = Zygote.withgradient(t -> f_adjoint(t, p_args), t_sample)
                l_tmp = res.val
                g_tmp = res.grad[1]
                gnorm = norm(g_tmp)

                is_good = (gnorm > 1e-8) && (l_tmp < initial_loss * 10)

                if is_good
                    push!(good_samples, (gnorm, l_tmp, copy(t_sample)))
                end

                if initialization_samples <= 50
                    println("Sample $s (mag=$(round(mag, sigdigits=3))): loss=$l_tmp grad_norm=$gnorm (GOOD: $is_good)")
                end
            end

            sort!(good_samples, by=x -> x[1], rev=true)
            top_n = min(5, length(good_samples))

            if top_n == 0
                println("No good samples found, falling back to random initialization")
                t_vals = real(collect(values(t_dict)))

            else
                best_t = nothing
                best_loss = Inf
                println("Performing quick optimization on top $top_n candidates...")
                quick_maxiters = min(30, maxiters)
                for i in 1:top_n
                    candidate_t = good_samples[i][3]
                    _, opt_t, opt_loss = execute_optimization(candidate_t, quick_maxiters, [:GradientDescent, :LBFGS])
                    println("Candidate $i quick opt loss: $opt_loss")
                    if opt_loss < best_loss
                        best_loss = opt_loss
                        best_t = opt_t
                    end
                end
                t_vals = best_t
                println("Selected best candidate with loss=$best_loss")
            end
        else
            error("DON'T DO THIS EVER")
        end

        println("Parameter count: $(length(t_vals))")

        _, t_vals, loss = execute_optimization(t_vals, maxiters, optimizers)
        coefficients = t_vals


        vals = update_values(signs, param_index_map, coefficients)

        # Construct and Store Matrix
        if antihermitian
            computed_matrices[order] = make_antihermitian(sparse(rows, cols, vals, dim, dim))
        else
            computed_matrices[order] = make_hermitian(sparse(rows, cols, vals, dim, dim))
        end


        println("Finished order $order")
        push!(metrics["loss"], loss)
        # push!(metrics["loss_std"], std(last(tmp_losses, 20)))
        computed_coefficients[order] = coefficients
        # Store back into cache
        # operator_cache[order][:coefficients] = coefficients

        # Metrics using all computed matrices
        # Need to clean list for metrics?
        clean_matrices = [m for m in computed_matrices if !isnothing(m)]
        for (k, func) in metric_functions
            push!(metrics[k], func(state1, state2, clean_matrices, tmp_losses))
        end
    end
    # display(plot(loss_tracker, yscale=:log10))
    # println("hey")
    return computed_matrices, coefficient_labels, computed_coefficients, metrics, operator_cache
end


# -----------------------------------------------------------------------------
# Optimized Adjoint Gradient Logic
# -----------------------------------------------------------------------------

"""
    adjoint_loss(t_vals, ops, rows, cols, signs, param_index_map, dim, v1, v2, p)

Computes the loss L = 1 - |<v1 | exp(im * (A(t) + p)) | v2>|^2 efficiently using expv
Custom rrule provides efficient gradients w.r.t t_vals.
"""
function adjoint_loss(t_vals, ops, rows, cols, signs, param_index_map, dim, v1, v2, p, antihermitian=false)
    # 1. Construct A(t) efficiently
    vals = update_values(signs, param_index_map, t_vals)
    A = sparse(rows, cols, vals, dim, dim)
    if antihermitian
        A = make_antihermitian(A)
    else
        A = make_hermitian(A)
    end

    # 2. Add offset p if present
    if !isnothing(p) && !(p isa SciMLBase.NullParameters)
        B = A + p
    else
        B = A
    end

    if antihermitian
        psi = expv(1.0, B, v2)
    else
        psi = expv(1.0im, B, v2)
    end

    # 4. Overlap
    # ov = <v1 | psi>
    overlap = dot(v1, psi)
    loss = 1 - abs2(overlap)
    # Zygote.@ignore println("loss=$loss coeff-mag=$(sum(abs.(t_vals)))")
    return loss
end

function ChainRulesCore.rrule(::typeof(adjoint_loss), t_vals, ops, rows, cols, signs, param_index_map, dim, v1, v2, p, antihermitian)
    # --- Reconstruct A ---
    vals = update_values(signs, param_index_map, t_vals)
    A = sparse(rows, cols, vals, dim, dim)
    if antihermitian
        A = make_antihermitian(A)
    else
        A = make_hermitian(A)
    end
    
    if !isnothing(p) && !(p isa SciMLBase.NullParameters)
        A = A + p
    end

    # --- Exact Gradient via Block Matrix ---
    # We want to compute: <v1 | \nabla(exp(im*A) * v2)
    # The gradient vector for parameter M_i is given by the top block of exp(im * [A M_i; 0 A]) * [0; v2].
    # We compute this for each parameter in parallel.

    # First, compute forward state and overlap for the primal return
    # Use tighter tolerance 1e-12 typically for unitarity, though default is ~1e-12.
    if antihermitian
        # psi, _ = exponentiate(A, 1.0, v2, tol=1e-12)
        psi = expv(1.0, A, v2)
    else
        # psi, _ = exponentiate(A, 1.0im, v2, tol=1e-12)
        psi = expv(1.0im, A, v2)
    end
    overlap = dot(v1, psi)
    y = 1 - abs2(overlap)

    function adjoint_loss_pullback(ȳ)
        # ȳ is sensitivity of loss (scalar)

        # dL/dt = -2 Re( <psi|v1> * <v1| dpsi/dt > )
        # We need <v1 | dpsi/dt > for each parameter.

        grad_t = Vector{Float64}(undef, length(t_vals))

        # Quadrature Gradient Implementation (N=50)
        N_steps = 50
        dt = 1.0 / N_steps

        # Forward Checkpoints: phis[k] ~ exp(i A t_k) v2
        # phis[1] corresponding to t=0 is v2.
        phis = Vector{Vector{ComplexF64}}(undef, N_steps + 1)
        phis[1] = v2

        for k in 1:N_steps
            if antihermitian
                # phis[k+1], _ = exponentiate(A, dt, phis[k], tol=1e-12)
                phis[k+1] = expv(dt, A, phis[k])
            else
                # phis[k+1], _ = exponentiate(A, dt * 1.0im, phis[k], tol=1e-12)
                phis[k+1] = expv(dt * 1.0im, A, phis[k])
            end
        end
        # Note: phis[end] should match 'psi' captured from primal pass.

        # Backward Checkpoints: chis[k] ~ exp(-i A (1-t_k)) v1
        # chis[N+1] corresponding to t=1 is v1.
        chis = Vector{Vector{ComplexF64}}(undef, N_steps + 1)
        chis[N_steps+1] = v1

        for k in N_steps:-1:1
            if antihermitian
                # chis[k], _ = exponentiate(A, -dt, chis[k+1], tol=1e-12)
                chis[k] = expv(-dt, A, chis[k+1])
            else
                # chis[k], _ = exponentiate(A, -dt * 1.0im, chis[k+1], tol=1e-12)
                chis[k] = expv(-dt * 1.0im, A, chis[k+1])
            end
        end

        # Simpson's Rule Weights
        weights = ones(N_steps + 1)
        weights[2:2:end-1] .= 4.0
        weights[3:2:end-2] .= 2.0
        weights[1] = 1.0
        weights[end] = 1.0
        weights .*= (dt / 3.0)

        # Pre-compute overlap factor (using captured 'overlap' from primal)
        conj_overlap_factor = conj(overlap) * ȳ

        # Accumulate gradients parallelized over parameters
        Threads.@threads for i in eachindex(grad_t)
            M = ops[i]
            val = 0.0 + 0.0im
            for k in 1:(N_steps+1)
                term = dot(chis[k], M, phis[k])
                val += term * weights[k]
            end

            # dO/dt = i * integral
            # dO/dt = int
            if antihermitian
                dO_dt = val
            else
                dO_dt = val * 1.0im
            end
            grad_t[i] = -2 * real(conj_overlap_factor * dO_dt) + 1e-3 * t_vals[i] # regularization
        end

        return NoTangent(), grad_t, NoTangent(), NoTangent(), NoTangent(), NoTangent(), NoTangent(), NoTangent(), NoTangent(), NoTangent(), NoTangent(), NoTangent(), NoTangent(), NoTangent(), NoTangent()
    end

    return y, adjoint_loss_pullback
end




In [42]:
# DO NOT MODIFY THIS CELL
"""
Wrapper function for optimize_unitary that will be used at the highest level. 
"""
function perform_optimization(degen_rm_U::Union{AbstractMatrix,Vector}, u_idx1::Int, u_idx2::Int, indexer::CombinationIndexer,
    spin_conserved::Bool=false; optimization_scheme::Vector{Int}=[1,2],antihermitian::Bool=false,
    maxiters=100, gradient::Symbol=:gradient, metric_functions::Dict{String,Function}=Dict{String,Function}(), optimizer::Union{Symbol,Vector{Symbol}}=:LBFGS,
    perturb_optimization::Float64=0.1,
    save_folder::Union{String,Nothing}=nothing, save_name::String="data"
)
    println("\n--- Optimizing between U indices: $u_idx1 and $u_idx2 ---")

    state1 = degen_rm_U[u_idx1, :]
    state2 = degen_rm_U[u_idx2, :]

    args = optimize_unitary(state1, state2, indexer;
        spin_conserved=spin_conserved,
        maxiters=maxiters, optimization_scheme=optimization_scheme, gradient=gradient,
        metric_functions=metric_functions, antihermitian=antihermitian, optimizer=optimizer,
        initial_coefficients=Any[], perturb_optimization=perturb_optimization)

    computed_matrices, coefficient_labels, coefficient_values, metrics, operator_cache = args

    data_dict = Dict{String,Any}(
        "all_matrices" => computed_matrices,
        "coefficients" => coefficient_values,
        "coefficient_labels" => coefficient_labels,
        "metrics" => metrics,
        "labels" => Dict(
            "starting state" => Dict("U index" => u_idx1),
            "ending state" => Dict("U index" => u_idx2)
        ),
        "states" => Dict(
            "starting state"=>state1,
            "ending state"=>state2,
        ),
        "operators" => operator_cache
    )

    # if !isnothing(save_folder)
    #     mkpath(save_folder)
    #     JLD2.jldsave(joinpath(save_folder, "$(save_name).jld2"); dict=data_dict)
    # end

    return data_dict
end


perform_optimization

In [59]:
# DON'T MODIFY THIS CELL

# collect necessary data from file
# folder = "/home/jek354/research/ML-signproblem/experimenting/ed/data/N=(3, 3)_3x2"
folder = "data/N=(4, 5)_3x3"
file_path = joinpath(folder, "meta_data_and_E.jld2")
dic = load_saved_dict(file_path)
meta_data = dic["meta_data"]
U_values = meta_data["U_values"]
all_full_eig_vecs = dic["all_full_eig_vecs"]
all_E = dic["E"] # Needed for energy selection
indexer = dic["indexer"]

# display meta data for logging
println("Meta data:")
display(meta_data)

# Extract N for saving
N = meta_data["electron count"]
spin_conserved = !isa(meta_data["electron count"], Number) # True if tuple (N_up, N_down)

# find lowest energy momentum sector and select it. 
min_E = Inf
k_min = 1
for (k, E_vec) in enumerate(all_E)
    if !isempty(E_vec)
        E_ground = E_vec[1]
        if E_ground < min_E
            min_E = E_ground
            k_min = k
        end
    end
end
println("Selected lowest energy symmetry sector: $k_min with Energy $(min_E)")

# Select the eigenvectors for this sector
# all_full_eig_vecs is a list of sectors. each sector is a list of vectors (per U).
target_vecs = all_full_eig_vecs[k_min]
if indexer isa Vector
    indexer = indexer[k_min]
end

# performs the optimization
time_elapsed = @elapsed begin 
    data_dict = perform_optimization(target_vecs, 1, 25, indexer,
        spin_conserved; optimization_scheme=[1,2],
        maxiters=20, gradient=:adjoint_gradient,
        optimizer=[:GradientDescent, :LBFGS, :GradientDescent, :LBFGS],
        save_folder=nothing, save_name="unitary_map_energy_N=$N")
end


Meta data:
Selected lowest energy symmetry sector: 4 with Energy -14.99997777778833


Dict{String, Any} with 8 entries:
  "U_values"       => [1.0e-5, 0.001, 0.0013434, 0.00180472, 0.00242446, 0.0032…
  "maxiters"       => 200
  "optimizer"      => "BFGS"
  "sites"          => "3x3"
  "solver"         => "Lanczos"
  "basis"          => "adiabatic"
  "bc"             => "periodic"
  "electron count" => (4, 5)

┌ Warning: saved type CombinationIndexer{Coordinate{2, Int64}} is missing field k in workspace type; reconstructing
└ @ JLD2 /Users/jonathonkambulow/.julia/packages/JLD2/hbsZG/src/data/reconstructing_datatypes.jl:197


MethodError: MethodError: no method matching perform_optimization(::Matrix{ComplexF64}, ::Int64, ::Int64, ::JLD2.ReconstructedMutable{Symbol("CombinationIndexer{Coordinate{2, Int64}}"), (:a, :comb_dict, :inv_comb_dict, :lattice_dims), NTuple{4, Any}}, ::Bool; optimization_scheme::Vector{Int64}, maxiters::Int64, gradient::Symbol, optimizer::Vector{Symbol}, save_folder::Nothing, save_name::String)
The function `perform_optimization` exists, but no method is defined for this combination of argument types.

Closest candidates are:
  perform_optimization(::Union{AbstractMatrix, Vector}, ::Int64, ::Int64, !Matched::CombinationIndexer, ::Bool; optimization_scheme, antihermitian, maxiters, gradient, metric_functions, optimizer, perturb_optimization, save_folder, save_name)
   @ Main ~/Library/CloudStorage/Dropbox/programming/cornell courses/research/experimenting/ed/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_W3sZmlsZQ==.jl:5
  perform_optimization(::Union{AbstractMatrix, Vector}, ::Int6

In [ ]:
# DO NOT MODIFY
# verifying accuracy (sanity checks)
state1 = data_dict["states"]["starting state"]
state2 = data_dict["states"]["ending state"]
U_state1 = expv(1im,sum(data_dict["all_matrices"]),state1)
@assert norm(U_state1) ≈ 1
@assert data_dict["metrics"]["loss"][end] ≈ 1 - abs2(state2'*U_state1)
for order=1:2
    t_keys = build_nth_order_operator(order, indexer; omit_H_conj=true, conserve_spin=spin_conserved, conserve_momentum=true)
    dim = length(indexer.inv_comb_dict)
    # create matrix operators to make gradient computation faster
    ops = []
    for k in collect(t_keys)
        _rows, _cols, _signs, _ = build_nth_order_sparse([k], indexer)
        push!(ops, make_hermitian(sparse(_rows, _cols, _signs, dim, dim)))
    end

    @assert sum(ops[k] * data_dict["coefficients"][order][k] for k in eachindex(data_dict["coefficients"][order])) ≈ data_dict["all_matrices"][order]
end

println("performance:")
println("Time: $time_elapsed")
println("Loss: $(data_dict["metrics"]["loss"][end])")




performance:
Time: 22.74191575
Loss: 2.917359794929464e-5
